In [22]:
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

LOAD THE DATASET HEREE

In [23]:
train_df = pd.read_csv("Output/train_dataset.csv")
test_df  = pd.read_csv("Output/teXt_dataset.csv")

# DATASE CLASSESS

In [ ]:
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt

all_classes = sorted(train_df["label"].unique())
class_to_idx = {c:i for i,c in enumerate(all_classes)}

class LandUse(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((128,128)),
            transforms.ToTensor()
        ])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.loc[idx,"image"]).convert("RGB")
        img = self.transform(img)
        label = class_to_idx[self.df.loc[idx,"label"]]
        return img, label

train_loader = DataLoader(LandUse(train_df), batch_size=32, shuffle=True)
test_loader  = DataLoader(LandUse(test_df), batch_size=32)

LOAD THE RESNET MODEL

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(all_classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 0.4699
Epoch 2/5, Loss: 0.2945
Epoch 3/5, Loss: 0.2434
Epoch 4/5, Loss: 0.2159
Epoch 5/5, Loss: 0.1781


#CHECL AACURACY  AND F1 SCORE


In [29]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")

print("Test Accuracy:", accuracy)
print("F1 Score:", f1)

Test Accuracy: 0.8402994385527136
F1 Score: 0.861906575998896


# Confusion matrix

In [28]:

cm = confusion_matrix(all_labels, all_preds)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=all_classes)
display.plot(xticks_rotation=45)
plt.title("Confusion Matrix")
plt.savefig("Output/confusion_matrix.png")
plt.close()
